# Project 3 — Python Data Analysis

**Dataset:** Omair Sales Data (1,998 transactions, 2004–2006)  
**Author:** Hussam Ali  
**Course:** Takniat Al-Oloum — Data Analysis Projects  
**Instructor:** Dr. Abdullah Al-Omair

---

## Workflow (mapped to assignment tasks)

1. **Load** the dataset using pandas
2. **Data cleaning** & preprocessing (IQR outliers, typo fixes, date repair)
3. **Transform** — feature engineering (Profit, Margin, time fields)
4. **Exploratory Data Analysis** (EDA)
5. **Statistical analysis** on Sales & Profit
6. **Visualizations** with matplotlib & seaborn
7. **Insights & conclusions**

> **Cleaning logic** mirrors the Power Query M code used in Power BI — same IQR outlier detection, same ratio-based imputation, same typo fixes. Numerical results match across both projects.

## 1. Load Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.family'] = 'DejaVu Sans'

DATA_PATH = 'OmairSalesDataProject.xlsx'
df_raw = pd.read_excel(DATA_PATH)
print(f'Loaded: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns')
df_raw.head()

In [ ]:
# Diagnostic check — identify data quality issues
print('=== Data types ===')
print(df_raw.dtypes)
print('\n=== Missing values ===')
print(df_raw.isna().sum())
print('\n=== Unique values per categorical column ===')
for col in ['City', 'Rep', 'Store', 'Prod']:
    vals = sorted(df_raw[col].dropna().unique())
    print(f'{col} ({len(vals)}): {vals}')
print('\n=== Date range ===')
print(f"Min: {df_raw['Date'].min()}  |  Max: {df_raw['Date'].max()}")
print('\n=== Cost issues ===')
print(f"Cost dtype: {df_raw['Cost'].dtype}  (should be number)")
print(f"Non-numeric Cost values: {(~df_raw['Cost'].apply(lambda x: isinstance(x, (int, float)))).sum()}")

### Issues identified

| # | Issue | Resolution |
|---|-------|-----------|
| 1 | `Cost` stored as `object` (mixed text/number) | Coerce to numeric |
| 2 | `Cost = 50,000,000` outlier (data entry error) | IQR-based detection + imputation |
| 3 | Some `Cost` values are null or zero | Impute using `Sale × Ratio` |
| 4 | A few dates show years 2014–2015 (outside 2004–2006 range) | Subtract 10 years |
| 5 | `Rep` has `'Mjeeed'` (typo) and `'Mjeed'` | Unify to `'Mjeed'` |
| 6 | `Store` has `'Lulu'` and `'Lulu Hyper'` (variants) | Unify to `'Lulu'` |
| 7 | `Profit` column missing | Compute `Sale - Cost` |

## 2. Data Cleaning

### Architectural approach

The cleaning logic mirrors the **Power Query M code** used in the Power BI project. Key design decisions:

1. **IQR-based outlier detection** (not hard-coded thresholds) — adapts to the actual data distribution
2. **Cost imputation via `Sale × Ratio`** — preserves natural variability, scales per-row with Sale
3. **`CostFlag` audit column** — records which rows were modified and why (production-ready data lineage)
4. **Ratio computed from clean rows only** — prevents outliers from contaminating the imputation baseline

In [ ]:
df = df_raw.copy()

# ===== Step 1: Cost → numeric (text values become NaN) =====
df['Cost'] = pd.to_numeric(df['Cost'], errors='coerce')
print(f"After numeric coercion — Cost NaN: {df['Cost'].isna().sum()}")

In [ ]:
# ===== Step 2: IQR-based outlier detection on Cost =====
Q1 = df['Cost'].quantile(0.25)
Q3 = df['Cost'].quantile(0.75)
IQR = Q3 - Q1
LowerBound = Q1 - 1.5 * IQR
UpperBound = Q3 + 1.5 * IQR

print(f'Q1 = {Q1:,.2f}')
print(f'Q3 = {Q3:,.2f}')
print(f'IQR = {IQR:,.2f}')
print(f'Bounds: [{LowerBound:,.2f}, {UpperBound:,.2f}]')

In [ ]:
# ===== Step 3: Compute Cost/Sale ratio from CLEAN rows only =====
# A 'clean' row has: Cost is not null/zero, Cost within IQR bounds, Sale is positive
clean_mask = (
    df['Cost'].notna() & (df['Cost'] != 0) &
    (df['Cost'] >= LowerBound) & (df['Cost'] <= UpperBound) &
    df['Sale'].notna() & (df['Sale'] > 0)
)
clean = df[clean_mask]
Ratio = clean['Cost'].sum() / clean['Sale'].sum()
print(f'Clean rows: {len(clean):,} / {len(df):,}')
print(f'Sum Cost (clean): {clean["Cost"].sum():,.2f}')
print(f'Sum Sale (clean): {clean["Sale"].sum():,.2f}')
print(f'Ratio = Cost/Sale = {Ratio:.6f}  ({Ratio*100:.2f}%)')

In [ ]:
# ===== Step 4: Create CostFlag BEFORE replacing (audit column) =====
conditions = [
    df['Cost'].isna(),
    df['Cost'] == 0,
    (df['Cost'] < LowerBound) | (df['Cost'] > UpperBound),
]
choices = ['Imputed (missing)', 'Imputed (zero)', 'Imputed (outlier)']
df['CostFlag'] = np.select(conditions, choices, default='OK')

print('CostFlag distribution:')
print(df['CostFlag'].value_counts())
print(f'\nTotal modified rows: {(df["CostFlag"] != "OK").sum()}')

In [ ]:
# ===== Step 5: Replace problematic Cost values with Sale × Ratio =====
problematic = (
    df['Cost'].isna() | (df['Cost'] == 0) |
    (df['Cost'] < LowerBound) | (df['Cost'] > UpperBound)
)
df.loc[problematic, 'Cost'] = (df.loc[problematic, 'Sale'] * Ratio).round(2)
df['Cost'] = df['Cost'].round(2)

print(f'After imputation — Cost NaN: {df["Cost"].isna().sum()}')
print(f'New Cost range: [{df["Cost"].min():,.2f}, {df["Cost"].max():,.2f}]')

In [ ]:
# ===== Step 6: Fix date outliers (Year > 2010 → subtract 10 years) =====
df['Date'] = pd.to_datetime(df['Date'])
date_outliers = df['Date'].dt.year > 2010
print(f'Date outliers (Year > 2010): {date_outliers.sum()}')
if date_outliers.any():
    print('Before fix:')
    print(df.loc[date_outliers, ['Date', 'City', 'Cost', 'Sale']].head())
    df.loc[date_outliers, 'Date'] = df.loc[date_outliers, 'Date'] - pd.DateOffset(years=10)
    print(f'\nAfter fix — Date range: {df["Date"].min().date()} → {df["Date"].max().date()}')

In [ ]:
# ===== Step 7: Trim & Proper-case text fields =====
for col in ['City', 'Rep', 'Store', 'Prod']:
    df[col] = df[col].str.strip().str.title()

# ===== Step 8: Fix typos / variants =====
df['Rep'] = df['Rep'].str.replace('Mjeeed', 'Mjeed', regex=False)
df['Store'] = df['Store'].str.replace('Lulu Hyper', 'Lulu', regex=False)

print('After cleaning:')
for col in ['City', 'Rep', 'Store', 'Prod']:
    print(f'  {col}: {sorted(df[col].unique())}')

In [ ]:
# ===== Step 9: Rename Prod → Product, round Sale =====
df['Sale'] = df['Sale'].round(2)
df = df.rename(columns={'Prod': 'Product'})
print(f'Columns after renaming: {list(df.columns)}')

## 3. Feature Engineering (Transform)

In [ ]:
# Derived columns: Profit, Margin, time fields
df['Profit'] = (df['Sale'] - df['Cost']).round(2)
df['Margin'] = np.where(df['Sale'] == 0, np.nan, df['Profit'] / df['Sale'])
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['MonthName'] = df['Date'].dt.strftime('%b')
df['Quarter'] = 'Q' + df['Date'].dt.quarter.astype(str)
df['YearMonth'] = df['Date'].dt.to_period('M').astype(str)

# Reorder columns (matches Power BI model)
df = df[['Date', 'City', 'Rep', 'Store', 'Product',
         'Cost', 'Sale', 'Profit', 'Margin',
         'Year', 'Month', 'MonthName', 'Quarter', 'YearMonth',
         'CostFlag']]

# Sort by Date descending (matches M code)
df = df.sort_values('Date', ascending=False).reset_index(drop=True)

print(f'Final shape: {df.shape}')
df.head()

In [ ]:
# Save cleaned dataset for reuse in SQL project
df.to_csv('OmairSales_Clean.csv', index=False, encoding='utf-8-sig')
print(f'Saved: OmairSales_Clean.csv ({len(df):,} rows)')

### Sanity check — verify numerical match with Power BI dashboard

In [ ]:
expected = {
    'Average of Sale': 5042.72,
    'Min of Sale': 12.56,
    'Max of Sale': 9994.24,
    'StDev of Sale': 2875.59,
}
actual = {
    'Average of Sale': df['Sale'].mean(),
    'Min of Sale': df['Sale'].min(),
    'Max of Sale': df['Sale'].max(),
    'StDev of Sale': df['Sale'].std(),
}

print(f'{"Metric":<20} {"Python":>15} {"Power BI":>15} {"Match":>8}')
print('-' * 60)
for k in expected:
    match = '✓' if abs(actual[k] - expected[k]) < 0.5 else '✗'
    print(f'{k:<20} {actual[k]:>15,.2f} {expected[k]:>15,.2f} {match:>8}')

## 4. Exploratory Data Analysis (EDA)

In [ ]:
# Top-line KPIs (match Power BI dashboard)
print('=' * 50)
print('OMAIR SALES — KEY METRICS')
print('=' * 50)
kpis = {
    'Total Sales':     df['Sale'].sum(),
    'Total Cost':      df['Cost'].sum(),
    'Total Profit':    df['Profit'].sum(),
    'Profit Margin %': df['Profit'].sum() / df['Sale'].sum() * 100,
    'Transactions':    len(df),
    'Avg Sale':        df['Sale'].mean(),
    'Median Sale':     df['Sale'].median(),
    'Min Sale':        df['Sale'].min(),
    'Max Sale':        df['Sale'].max(),
    'StDev Sale':      df['Sale'].std(),
}
for k, v in kpis.items():
    if 'Margin' in k:
        print(f'  {k:<18} {v:>15,.2f}%')
    elif k == 'Transactions':
        print(f'  {k:<18} {int(v):>15,}')
    else:
        print(f'  {k:<18} {v:>15,.2f}')

In [ ]:
# 4.1 — Breakdown by City
city_summary = df.groupby('City').agg(
    TotalSales=('Sale', 'sum'),
    TotalProfit=('Profit', 'sum'),
    AvgSale=('Sale', 'mean'),
    Transactions=('Sale', 'count'),
).round(2).sort_values('TotalProfit', ascending=False)
city_summary['Margin%'] = (city_summary['TotalProfit'] / city_summary['TotalSales'] * 100).round(2)
print('=== Performance by City ===')
city_summary

In [ ]:
# 4.2 — Breakdown by Product
prod_summary = df.groupby('Product').agg(
    TotalSales=('Sale', 'sum'),
    TotalProfit=('Profit', 'sum'),
    AvgSale=('Sale', 'mean'),
    Transactions=('Sale', 'count'),
).round(2).sort_values('TotalProfit', ascending=False)
prod_summary['Margin%'] = (prod_summary['TotalProfit'] / prod_summary['TotalSales'] * 100).round(2)
print('=== Performance by Product ===')
prod_summary

In [ ]:
# 4.3 — Breakdown by Store
store_summary = df.groupby('Store').agg(
    TotalSales=('Sale', 'sum'),
    TotalProfit=('Profit', 'sum'),
    AvgSale=('Sale', 'mean'),
    Transactions=('Sale', 'count'),
).round(2).sort_values('TotalProfit', ascending=False)
store_summary['Margin%'] = (store_summary['TotalProfit'] / store_summary['TotalSales'] * 100).round(2)
print('=== Performance by Store ===')
store_summary

In [ ]:
# 4.4 — Breakdown by Rep
rep_summary = df.groupby('Rep').agg(
    TotalSales=('Sale', 'sum'),
    TotalProfit=('Profit', 'sum'),
    AvgSale=('Sale', 'mean'),
    Transactions=('Sale', 'count'),
).round(2).sort_values('TotalProfit', ascending=False)
rep_summary['Margin%'] = (rep_summary['TotalProfit'] / rep_summary['TotalSales'] * 100).round(2)
print('=== Performance by Sales Rep ===')
rep_summary

## 5. Statistical Analysis

In [ ]:
# Descriptive statistics on financial columns
stats_table = df[['Cost', 'Sale', 'Profit', 'Margin']].describe().round(2)
print('=== Descriptive Statistics ===')
stats_table

In [ ]:
# Coefficient of Variation (relative variability)
print('=== Coefficient of Variation (std / mean × 100) ===')
for col in ['Sale', 'Profit', 'Cost']:
    cv = df[col].std() / df[col].mean() * 100
    print(f'  CV({col}) = {cv:.2f}%')

# Correlation matrix
print('\n=== Correlation Matrix ===')
corr = df[['Cost', 'Sale', 'Profit', 'Margin']].corr().round(3)
corr

In [ ]:
# IQR-based outlier detection on Sale (post-cleaning sanity check)
Q1_s = df['Sale'].quantile(0.25)
Q3_s = df['Sale'].quantile(0.75)
IQR_s = Q3_s - Q1_s
low, high = Q1_s - 1.5*IQR_s, Q3_s + 1.5*IQR_s
outliers = df[(df['Sale'] < low) | (df['Sale'] > high)]
print(f'Sale IQR bounds: [{low:,.2f}, {high:,.2f}]')
print(f'Sale outliers: {len(outliers)} ({len(outliers)/len(df)*100:.2f}% of rows)')
print('\nInterpretation: Few outliers indicate the cleaning was effective.')

In [ ]:
# Year-over-year growth analysis
yearly = df.groupby('Year').agg(
    TotalSales=('Sale', 'sum'),
    TotalProfit=('Profit', 'sum'),
    Transactions=('Sale', 'count')
).round(2)
yearly['Sales_YoY%'] = (yearly['TotalSales'].pct_change() * 100).round(2)
yearly['Profit_YoY%'] = (yearly['TotalProfit'].pct_change() * 100).round(2)
print('=== Year-over-Year Growth ===')
yearly

## 6. Visualizations

### 6.1 — Performance Overview (4-panel)

In [ ]:
# Color palette matching Power BI dashboard
C_TEAL = '#3FA796'
C_ORANGE = '#E67E22'
C_NAVY = '#1E3A5F'
C_GREEN = '#2E7D32'
C_PURPLE = '#6A1B9A'
C_AMBER = '#E65100'

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Omair Sales — Performance Overview (2004–2006)',
             fontsize=15, fontweight='bold', color=C_NAVY)

# 1. Profit by City
ax = axes[0, 0]
city_summary['TotalProfit'].plot(kind='bar', ax=ax, color=C_TEAL,
                                   edgecolor='black', linewidth=0.5)
ax.set_title('Profit by City', fontweight='bold')
ax.set_ylabel('Profit (SAR)')
ax.tick_params(axis='x', rotation=30)
for i, v in enumerate(city_summary['TotalProfit']):
    ax.text(i, v + 15000, f'{v/1000:.0f}K', ha='center', fontsize=9)

# 2. Sales Share by Product (pie)
ax = axes[0, 1]
colors_pie = [C_TEAL, C_ORANGE, C_NAVY, C_AMBER]
ax.pie(prod_summary['TotalSales'], labels=prod_summary.index,
       autopct='%1.1f%%', startangle=90, colors=colors_pie)
ax.set_title('Sales Share by Product', fontweight='bold')

# 3. Profit by Store (horizontal)
ax = axes[1, 0]
store_summary['TotalProfit'].plot(kind='barh', ax=ax, color=C_ORANGE,
                                    edgecolor='black', linewidth=0.5)
ax.set_title('Profit by Store', fontweight='bold')
ax.set_xlabel('Profit (SAR)')
for i, v in enumerate(store_summary['TotalProfit']):
    ax.text(v + 15000, i, f'{v/1000:.0f}K', va='center', fontsize=9)

# 4. Profit by Sales Rep
ax = axes[1, 1]
rep_summary['TotalProfit'].plot(kind='bar', ax=ax, color=C_NAVY,
                                  edgecolor='black', linewidth=0.5)
ax.set_title('Profit by Sales Rep', fontweight='bold')
ax.set_ylabel('Profit (SAR)')
ax.tick_params(axis='x', rotation=30)
for i, v in enumerate(rep_summary['TotalProfit']):
    ax.text(i, v + 12000, f'{v/1000:.0f}K', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('viz_01_overview.png', dpi=100, bbox_inches='tight')
plt.show()

### 6.2 — Monthly Sales & Profit Trend

In [ ]:
monthly = df.groupby('YearMonth').agg(
    Sales=('Sale', 'sum'),
    Profit=('Profit', 'sum')
).reset_index()
monthly['YearMonth'] = pd.to_datetime(monthly['YearMonth'])
monthly = monthly.sort_values('YearMonth')

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(monthly['YearMonth'], monthly['Sales'], marker='o',
        label='Sales', linewidth=2.5, color=C_TEAL)
ax.plot(monthly['YearMonth'], monthly['Profit'], marker='s',
        label='Profit', linewidth=2.5, color=C_ORANGE)
ax.fill_between(monthly['YearMonth'], monthly['Profit'], alpha=0.15, color=C_ORANGE)

ax.set_title('Monthly Sales & Profit Trend (2004–2006)',
             fontsize=14, fontweight='bold', color=C_NAVY)
ax.set_xlabel('Month')
ax.set_ylabel('Amount (SAR)')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('viz_02_trend.png', dpi=100, bbox_inches='tight')
plt.show()

### 6.3 — Profit Heatmap: City × Product

In [ ]:
pivot = df.pivot_table(values='Profit', index='City', columns='Product', aggfunc='sum')

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(pivot, annot=True, fmt=',.0f', cmap='YlOrRd',
            linewidths=0.5, cbar_kws={'label': 'Profit (SAR)'}, ax=ax)
ax.set_title('Profit Heatmap — City × Product',
             fontsize=14, fontweight='bold', color=C_NAVY)
plt.tight_layout()
plt.savefig('viz_03_heatmap.png', dpi=100, bbox_inches='tight')
plt.show()

### 6.4 — Sale Distribution & Outlier Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Histogram with mean/median markers
ax = axes[0]
ax.hist(df['Sale'], bins=40, color=C_TEAL, edgecolor='black', alpha=0.75)
ax.axvline(df['Sale'].mean(), color='red', linestyle='--', linewidth=2,
           label=f'Mean: {df["Sale"].mean():,.0f}')
ax.axvline(df['Sale'].median(), color='blue', linestyle='--', linewidth=2,
           label=f'Median: {df["Sale"].median():,.0f}')
ax.set_title('Distribution of Sale Amounts', fontweight='bold')
ax.set_xlabel('Sale (SAR)')
ax.set_ylabel('Frequency')
ax.legend()

# Boxplot by Product
ax = axes[1]
products_ordered = sorted(df['Product'].unique())
df.boxplot(column='Sale', by='Product', ax=ax, grid=False)
ax.set_title('Sale Distribution by Product', fontweight='bold')
ax.set_xlabel('Product')
ax.set_ylabel('Sale (SAR)')
plt.suptitle('')  # remove default suptitle

plt.tight_layout()
plt.savefig('viz_04_distribution.png', dpi=100, bbox_inches='tight')
plt.show()

### 6.5 — Cost vs Sale Scatter (Profit colored)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
sc = ax.scatter(df['Cost'], df['Sale'], c=df['Profit'], cmap='RdYlGn',
                alpha=0.6, s=30, edgecolors='black', linewidth=0.3)
plt.colorbar(sc, label='Profit (SAR)')
ax.plot([0, df['Cost'].max()], [0, df['Cost'].max()], 'k--',
        alpha=0.5, label='Break-even (Cost = Sale)')
ax.set_title('Cost vs Sale (color = Profit)', fontsize=14,
             fontweight='bold', color=C_NAVY)
ax.set_xlabel('Cost (SAR)')
ax.set_ylabel('Sale (SAR)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('viz_05_scatter.png', dpi=100, bbox_inches='tight')
plt.show()

### 6.6 — Top Sales Rep per Product (matches SQL Q2)

In [ ]:
# Top rep per product — same logic as SQL Q2
rep_prod = df.groupby(['Product', 'Rep'])['Profit'].sum().reset_index()
top_rep = rep_prod.loc[rep_prod.groupby('Product')['Profit'].idxmax()]
top_rep = top_rep.sort_values('Profit', ascending=False)

print('=== Top Sales Rep per Product Category ===')
print(top_rep.to_string(index=False))

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(top_rep['Product'] + '\n(' + top_rep['Rep'] + ')',
              top_rep['Profit'], color=C_TEAL, edgecolor='black')
ax.set_title('Top Sales Rep per Product Category',
             fontsize=14, fontweight='bold', color=C_NAVY)
ax.set_ylabel('Profit (SAR)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))
for bar, v in zip(bars, top_rep['Profit']):
    ax.text(bar.get_x() + bar.get_width()/2, v + 5000,
            f'{v/1000:.0f}K', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('viz_06_top_reps.png', dpi=100, bbox_inches='tight')
plt.show()

## 7. Insights & Conclusions

### Key Findings

**1. Business scale & profitability**  
The dataset captures **10,075,360.98 SAR** in sales and **3,796,445.84 SAR** in profit across **1,998 transactions** (Jun 2004 – Jun 2006). Profit margin is a healthy **37.68%**, with an average ticket size of **5,042.72 SAR**.

**2. Annual growth — strong but uneven**  
- 2004 (partial, from Jun): 2,756,652.95 SAR  
- 2005 (full year): **5,085,395.93 SAR** — +84% YoY growth  
- 2006 (partial, ends Jun): 2,233,312.10 SAR  
Annualized, the business shows healthy expansion, but the partial-year edges require careful interpretation.

**3. Geographic balance — no over-dependence**  
Profit is distributed across six cities with **only ~30% spread** between top (Riyadh: 715,437 SAR) and bottom (Hail: 552,446 SAR). No city is critically dependent — a resilient distribution that reduces single-market risk.

**4. Product portfolio — low concentration risk**  
All four product categories (Food, Office, Toys, Home) contribute similar profit (range: **864K–1.02M SAR**), with margins clustered tightly at **37%–38%**. Food leads marginally, but no single product carries the business.

**5. Sales rep performance — Ghalia dominates 3 of 4 categories**  
- **Food** → Ghalia (230,868 SAR)  
- **Toys** → Ghalia (226,749 SAR)  
- **Home** → Maliha (184,763 SAR)  
- **Office** → Yahia (249,543 SAR)  

The gap between top and bottom rep is **under 20%**, suggesting standardized training and consistent execution across the team.

**6. Store consolidation impact**  
After unifying `Lulu Hyper` → `Lulu`, **Lulu emerges as the leading retail channel**. Without this data fix, store rankings would have been split and misleading — a clear case where **data cleaning directly affects business decisions**.

**7. MoM volatility — base effects detected**  
Cross-validation with SQL Q3 reveals **9 extreme MoM spikes (>200%)**, all driven by low prior-month bases rather than genuine momentum (e.g., Jubail Dec 2004: +1,384% after a 2K SAR November). Median MoM is **-5.6%**, indicating sales are reactive rather than smoothly planned.

### Data Quality Lessons

The raw dataset contained:
- **One extreme outlier** (`Cost = 50,000,000`) — would have skewed averages 5x
- **Three date entry errors** (years 2014–2015 instead of 2004–2005)
- **Typo inconsistencies**: `Mjeed` vs `Mjeeed`, `Lulu` vs `Lulu Hyper`
- **Type confusion**: `Cost` stored as string instead of numeric

**Architectural lesson**: IQR-based outlier detection **adapts to actual data distribution** and is preferable to hard-coded thresholds. The `CostFlag` audit column preserves data lineage for compliance and debugging.

### Recommendations

**Operational**  
Investigate why **Hail consistently lags** (552K vs Riyadh's 715K profit — a 22% gap). Possible drivers: smaller addressable market, weaker rep allocation, fewer store locations. Replicate Riyadh's playbook to lift Hail.

**Process**  
Add **input validation at the source** to prevent future entries like `Cost = 50,000,000`:
- Reject Cost > 10× median Cost per Product
- Enforce date window (2004-01-01 → current)
- Use dropdowns for City/Rep/Store to eliminate typos

**Analytics**  
Migrate the Power BI dashboard to **Power BI Service** for scheduled refresh, shared access, and stakeholder subscriptions.

**Pipeline**  
The same cleaning logic is implemented across **Power Query (M)**, **Python (pandas)**, and **SQL Server (T-SQL)** — three implementations of the same algorithm. Promote it to a single reusable specification (e.g., dbt model or Python module) to eliminate triplication risk.

### Cross-Tool Validation

All four analytical tools (Excel, Power BI, Python, SQL Server) produce **identical numerical results** (Total Sales = 10,075,360.98 SAR ± rounding). This cross-validation provides high confidence in the findings.